##Libraries

In [ ]:
# Import general utilities
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
# from google.colab import drive

# Import TDA utilities
import gudhi
import gudhi.representations
from ripser import Rips
from persim.landscapes import PersLandscapeExact
from persim.landscapes import plot_landscape_simple
from persim.landscapes import PersistenceLandscaper
from persim import plot_diagrams
from tadasets import torus, sphere

# Import Scikit-Learn tools
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn import metrics
from sklearn.model_selection import train_test_split
from mpl_toolkits.mplot3d import Axes3D # Required for 3D plots


##Handwriten Sample - Cher

In [ ]:
from scipy.spatial import transform
# Example handwritten by Cher.
sample_diagram = [
    np.array([[1,5], [2,6], [3,4]])
]
# Plotting objects
fig, axs = plt.subplots(1, 3)
fig.set_size_inches(16, 8)
# TDA Creating Persistence Landscape
landscape = PersLandscapeExact(dgms = sample_diagram, hom_deg=0)

# Plotting Peristence Diagram
axs[0].scatter([t[0] for t in sample_diagram[0]], [t[1] for t in sample_diagram[0]], label="H0", color="black")
diagonal_line_x = np.linspace(0, 6.5, 100)
axs[0].plot(diagonal_line_x, diagonal_line_x, color='black')
# Fill the area below the diagonal line in grey
axs[0].fill_between(diagonal_line_x, 0, diagonal_line_x, color='lightgray', alpha=0.5)
axs[0].set_title("Persistence Diagram", fontsize=20)
axs[0].set_xlabel("Birth", fontsize=20)
axs[0].set_ylabel("Death", fontsize=20)
axs[0].set_xlim([0, 6.5])
axs[0].set_ylim([0, 6.5])
axs[0].legend(fontsize=20, title="Dimension", loc='lower right') # Increased fontsize to 20

# Plotting Piecewise Peristence Functions
axs[1].set_title("Piecewise Peristence Functions", fontsize=20)
axs[1].set_xlabel("Value", fontsize=20)
axs[1].set_ylabel("Persistence", fontsize=20)
axs[1].set_xlim([-.5, 6.5])
axs[1].set_ylim([-.5, 6.5])

midpt = lambda birth, death: (birth + death) / 2
lifespan = lambda birth, death: death - birth

# Correctly generate x and y coordinates for each tent function
all_l_x = []
all_l_y = []

for i in range(len(sample_diagram[0])):
    b, d = sample_diagram[0][i]
    mid_point = midpt(b, d)
    life_span = lifespan(b, d)

    # Coordinates for one tent function, extended to the plotting limits if necessary
    # This matches the (birth,0) -> (midpoint, lifespan) -> (death,0) pattern.
    x_tent = [0, b, mid_point, d, 6.6] # Extend to 0 and 6.6 for full range visibility
    y_tent = [0, 0, life_span, 0, 0]

    all_l_x.append(x_tent)
    all_l_y.append(y_tent)

for i in range(len(all_l_x)):
  axs[1].plot(all_l_x[i], all_l_y[i], label=f"Tent {i+1}")
  axs[1].fill_between(all_l_x[i], 0, all_l_y[i], color='lightblue', alpha=0.3)

# Plot Persistence Landscape
axs[2].set_title("Persistence Landscape", fontsize=20)
plot_landscape_simple(
    landscape,
    ax=axs[2],
)


for i in range(0,3):
  axs[i].grid(True)
  axs[i].set_aspect('equal', adjustable='box')
  axs[i].set_xlim([-.5, 6.5])
  axs[i].set_ylim([-.5, 6.5])
  axs[i].legend(fontsize=10) # Increased fontsize to 20

##Creating Sample Data: Circle, 1D double Tourus, Torus, and Sphere

In [ ]:
num_points = 100
radius = 1
noise_level = .1

theta = np.linspace(0, 2 * np.pi, num_points)
x_perfect = radius * np.cos(theta)
y_perfect = radius * np.sin(theta)

x_noise = np.random.uniform(-noise_level, noise_level, num_points)
y_noise = np.random.uniform(-noise_level, noise_level, num_points)

x_approx = x_perfect + x_noise
y_approx = y_perfect + y_noise

Circle_data = np.column_stack((x_approx, y_approx))


# Creating Double Torus 2D datacloud points.
num_points_eight = 100
noise_level_eight = 0.1 # Small noise to make it realistic

t = np.linspace(-np.pi, np.pi, num_points_eight)
x_eight = 2*(np.sin(t))
y_eight = 2*(np.cos(t)) * (np.sin(t))


x_noise_eight = np.random.normal(0, noise_level_eight, num_points_eight)
y_noise_eight = np.random.normal(0, noise_level_eight, num_points_eight)

DoubleTorus2D_data = np.column_stack((x_eight + x_noise_eight, y_eight + y_noise_eight))

# Creating Sphere datacloud points with noise.
num_points_sphere = 100
radius_sphere = 1
noise_level_sphere = .1
Sphere_data = sphere(n=num_points_sphere, r=radius_sphere, noise=noise_level_sphere)


# Creating 3D Torus datacloud points with noice.
num_points_torus = 100
c_torus = 1 # Major axis
a_torus = 1 # Minor axis

noise_level_torus = 0.1 # Small noise to make it realistic but retain shape

Torus_data = torus(n=num_points_torus, c=c_torus, a=a_torus, noise=noise_level_torus)

##Visualizing the Data

In [ ]:

fig = plt.figure(figsize=(12, 8), dpi=159)

# Subplot 1: Circular Data (2D)
ax1 = fig.add_subplot(2, 2, 1)
ax1.scatter(Circle_data[:, 0], Circle_data[:, 1], alpha=0.7)
ax1.set_title("Circular Data", fontsize=20) # Increased fontsize to 20

ax1.set_aspect('equal', adjustable='box')
ax1.grid(True)

# Subplot 2: Double Torus 2D Data (Figure-Eight)
ax2 = fig.add_subplot(2, 2, 2)
ax2.scatter(DoubleTorus2D_data[:, 0], DoubleTorus2D_data[:, 1], alpha=0.7)
ax2.set_title("Double Torus 2D Data (Figure-Eight)", fontsize=20) # Increased fontsize to 20
ax2.set_aspect('equal', adjustable='box')
ax2.grid(True)

# Subplot 3: Sphere Data (3D)
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
ax3.scatter(Sphere_data[:, 0], Sphere_data[:, 1], Sphere_data[:, 2], alpha=0.7)
ax3.set_title("Sphere Data (3D)", fontsize=20) # Increased fontsize to 20

# Subplot 4: Torus Data (3D)
ax4 = fig.add_subplot(2, 2, 4, projection='3d')
ax4.scatter(Torus_data[:, 0], Torus_data[:, 1], Torus_data[:, 2], alpha=0.7)
ax4.set_title("Torus Data (3D)", fontsize=20) # Increased fontsize to 20


plt.tight_layout()
plt.show()

## Persistance Diagram and Persistence Landscape

In [ ]:
rip = Rips(maxdim=2);
dmg_Circular_data = rip.fit_transform(Circle_data)


fig = plt.figure(figsize=(15, 10))
# Plot Data
ax0 = fig.add_subplot(2, 2, 1)
ax1 = fig.add_subplot(2, 2, 2)
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, 4)

ax0.scatter(Circle_data[:, 0], Circle_data[:, 1], alpha=0.7)
# Plot Persistence Diagrams
plot_diagrams(dmg_Circular_data, title="Persistence Diagram Circle", ax=ax1)
#Plot Persistence Landscape
plot_landscape_simple(
    PersLandscapeExact(dmg_Circular_data, hom_deg=0),
    title="Degree 1 Persistence Landscape Circle",
    ax=ax2,
)
plot_landscape_simple(
    PersLandscapeExact(dmg_Circular_data, hom_deg=1),
    title="Degree 2 Persistence Landscape Circle",
    ax=ax3,
)

ax2.get_legend().remove()
ax3.get_legend().remove()

## Loading Epha2a Data and control

##TDA Analysis:

In [ ]:
import pandas as pd
import numpy as np
epha2a = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")


x_epha2a = epha2a['Center of the object_0']
y_epha2a = epha2a['Center of the object_1']
z_epha2a = epha2a['Center of the object_2']

x_control = control['Center of the object_0']
y_control = control['Center of the object_1']
z_control = control['Center of the object_2']

epha2_data = np.column_stack((x_epha2a, y_epha2a, z_epha2a))
control_data = np.column_stack((x_control, y_control, z_control))

##Creating Persistance Landscapes Epha2a

In [ ]:
dgm_epha2_data       = Rips(maxdim=2).fit_transform(epha2_data)
dgm_control_data     = Rips(maxdim=2).fit_transform(control_data)


fig = plt.figure(figsize=(10,10), dpi=159)
# Plot Data

#3D plots
ax0 = fig.add_subplot(1, 2, 1, projection='3d')
ax1 = fig.add_subplot(1, 2, 2, projection='3d')

ax1.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:,2])
ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:,2])
ax1.set_title("Epha2a")
ax0.set_title("Control")
# Change rotation: 30 degrees elevation, 45 degrees azimuth
ax0.view_init(elev=45, azim=90, roll=0)
ax1.view_init(elev=45, azim=90, roll=0)
# Plot Persistence Diagrams
fig2 = plt.figure(figsize=(10,10), dpi=159)
ax2 = fig2.add_subplot(1, 2, 1)
ax3 = fig2.add_subplot(1, 2, 2)


plot_diagrams(dgm_control_data, title="Control", ax=ax2)
plot_diagrams(dgm_epha2_data, title="Epha2a", ax=ax3)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(15, 7))

# Subplot 1: Epha2a Data with Surface
ax1 = fig.add_subplot(122, projection='3d')
ax1.plot_trisurf(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], cmap='viridis', alpha=0.7)
ax1.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], color='red', s=20, alpha=0.9)
ax1.set_title('Epha2a Data with Surface', fontsize=16)
ax1.set_xlabel('X', fontsize=12)
ax1.set_ylabel('Y', fontsize=12)
ax1.set_zlabel('Z', fontsize=12)
ax1.view_init(elev=35, azim=90, roll=0)
ax1.set_box_aspect([1, 1, 1])

# Subplot 2: Control Data with Surface
ax2 = fig.add_subplot(121, projection='3d')
ax2.plot_trisurf(control_data[:, 0], control_data[:, 1], control_data[:, 2], cmap='viridis', alpha=0.7)
ax2.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2], color='blue', s=20, alpha=0.9)
ax2.set_title('Control Data with Surface', fontsize=16)
ax2.set_xlabel('X', fontsize=12)
ax2.set_ylabel('Y', fontsize=12)
ax2.set_zlabel('Z', fontsize=12)
ax2.view_init(elev=35, azim=45, roll=0)
ax2.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.show()

###3D Graphs

In [ ]:
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider
    from IPython.display import display
except ImportError:
    print("ipywidgets not found. Installing...")
    %pip install ipywidgets
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider
    from IPython.display import display


@interact(elev=IntSlider(min=-90, max=90, step=5, value=35, description='Elevation:'),
          azim=IntSlider(min=0, max=360, step=5, value=90, description='Azimuth:'))
def plot_epha2a_interactive(elev, azim):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot_trisurf(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], cmap='viridis', alpha=0.7)
    ax.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2])

    ax.set_title(f'Epha2a Data with Interactive View (Elev: {elev}, Azim: {azim})', fontsize=16)
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=elev, azim=azim)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

@interact(elev=IntSlider(min=-90, max=90, step=5, value=35, description='Elevation:'),
          azim=IntSlider(min=0, max=360, step=5, value=45, description='Azimuth:'))
def plot_control_interactive(elev, azim):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot_trisurf(control_data[:, 0], control_data[:, 1], control_data[:, 2], cmap='viridis', alpha=0.7)
    ax.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2])

    ax.set_title(f'Control Data with Interactive View (Elev: {elev}, Azim: {azim})', fontsize=16)
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_box_aspect([1,1,1])
    ax.view_init(elev=elev, azim=azim)
    plt.show()

In [ ]:
dgm_Circle_data        = Rips(maxdim=2).fit_transform(epha2_data)
dgm_DoubleTorus2D_data = Rips(maxdim=2).fit_transform(DoubleTorus2D_data)
dgm_Sphere_data        = Rips(maxdim=2).fit_transform(Sphere_data)
dgm_Torus_data         = Rips(maxdim=2).fit_transform(Torus_data)

In [ ]:
cloud_data = []
cloud_data.append(dgm_Circle_data)
cloud_data.append(dgm_DoubleTorus2D_data)
cloud_data.append(dgm_Sphere_data)
cloud_data.append(dgm_Torus_data)


n_lambda = 10
remove_legend = True

fig, axs = plt.subplots( len(cloud_data), 3) # 4 Persistence Landscape and three properties ( H0 - H2)
fig.set_size_inches(12, 8)

all_ax = axs.flatten()

# Define column titles
shape         = ["Circle", "Double Torus", "Sphere", "Torus"]
column_titles = ["Connected", "Loops", "Void", "Hyper-Voids"]

for i, dgm_data in enumerate(cloud_data):
  for j in range(len(dgm_data)): # j is the homology dimension (0 for H0, 1 for H1, 2 for H2)
    current_diagram = dgm_data[j]

    # Filter out infinite points as they cannot form a proper landscape function with finite birth/death
    finite_points_in_diagram = np.array([p for p in current_diagram if p[1] != np.inf])

    # Only attempt to plot if there are finite points in the diagram for this homology dimension
    if finite_points_in_diagram.shape[0] == 0:
      continue # Skip plotting for empty or all-infinite diagrams

    plot_landscape_simple(
        PersLandscapeExact(dgm_data, hom_deg=j), # Pass the full list of diagrams
        ax=all_ax[i*3 + j], depth_range = range(n_lambda) )

    # all_ax[i*3+j].set_xlim([0, 2])
    # all_ax[i*3+j].set_ylim([0, 1])
    # Set column titles for the top row only
    if i == 0:
        all_ax[j].set_title(column_titles[j], fontsize=20) # Set title for the first row's subplots

    if ( remove_legend ):
      legend = all_ax[i*3 + j].get_legend()
      if legend:
        legend.remove()

all_ax[2].set_title("Void", fontsize=20)
all_ax[0].set_ylabel("Circle", fontsize=15)
all_ax[3].set_ylabel("Double Torus", fontsize=15)
all_ax[6].set_ylabel("Sphere", fontsize=15)
all_ax[9].set_ylabel("Torus", fontsize=15)
fig.tight_layout() # Adjust layout to provide spacing



In [ ]:
# Compute multiple persistence landscapes
rips = Rips(maxdim=2)
landscapes_torus = []
landscapes_sphere = []

for i in range(100):
    # Resample data
    _data_torus = torus(n=100, c=2, a=1)
    _data_sphere = sphere(n=100, r=2)

    # Compute persistence diagrams
    dgm_torus = rips.fit_transform(_data_torus)

    # Instantiate persistence landscape transformer
    torus_landscaper = PersistenceLandscaper(
        hom_deg=1, start=0, stop=2.0, num_steps=500, flatten=True
    )

    # Compute flattened persistence landscape
    torus_flat = torus_landscaper.fit_transform(dgm_torus)

    landscapes_torus.append(torus_flat)

    # Compute persistence diagrams
    dgm_sphere = rips.fit_transform(_data_sphere)

    # Instantiate persistence landscape transformer
    sphere_landscaper = PersistenceLandscaper(
        hom_deg=1, start=0, stop=2.0, num_steps=500, flatten=True
    )

    # Compute flattened persistence landscape
    sphere_flat = sphere_landscaper.fit_transform(dgm_sphere)

    landscapes_sphere.append(sphere_flat)

landscapes_torus = np.array(landscapes_torus, dtype=object)
landscapes_sphere = np.array(landscapes_sphere, dtype=object)
print("Torus:", np.shape(landscapes_torus))
print("Sphere:", np.shape(landscapes_sphere))


In [ ]:
# Find maximal length
u = np.max([len(a) for a in landscapes_torus])
v = np.max([len(a) for a in landscapes_sphere])

w = np.max([u, v])

# Instantiate zero-padded arrays
ls_torus = np.zeros((100, w))
ls_sphere = np.zeros((100, w))

# Populate arrays
for i in range(len(landscapes_torus)):
    ls_torus[i, 0 : len(landscapes_torus[i])] = landscapes_torus[i]
    ls_sphere[i, 0 : len(landscapes_sphere[i])] = landscapes_sphere[i]


print("Torus:", ls_torus.shape)
print("Sphere:", ls_sphere.shape)

In [ ]:
# Instantiate PCA solver
pca_torus = PCA(n_components=2)

# Compute PCA
pca_torus.fit_transform(ls_torus)

# Define components
comp_torus = pca_torus.components_

# Instantiate PCA solver
pca_sphere = PCA(n_components=2)

# Compute PCA
pca_sphere.fit_transform(ls_sphere)

# Define components
comp_sphere = pca_sphere.components_

In [ ]:
# Plot projection of data onto the first two principal components
plt.figure()
plt.scatter(comp_sphere[0], comp_sphere[1], label="Sphere", alpha=0.4)
plt.scatter(comp_torus[0], comp_torus[1], label="Torus", alpha=0.2)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()